# 23. 발표 이후 피드백: 단일 곡 AI 생성 여부 판별

사용자가 제공한 MP3 두 곡을 기존 프로젝트의 최종 튜닝 모델에 입력한다. 파일명이나 메타데이터는 특징으로 사용하지 않고, PCM 오디오만 사용한다. 모델은 10초 구간별 점수를 낸 뒤 곡 단위 평균으로 최종 판정한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (parent / "data").exists() and (parent / "src").exists():
            PROJECT_ROOT = parent
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_FILES = [
    Path.home() / "Downloads/fake_00001_suno_1.mp3",
    Path.home() / "Downloads/fake_00001_suno_0.mp3",
]
OUTPUT_DIR = PROJECT_ROOT / "results/post_presentation_feedback"

from src.single_track_inference import run_inference, save_results
results = run_inference(INPUT_FILES, PROJECT_ROOT)
save_results(results, OUTPUT_DIR)
print("저장 위치:", OUTPUT_DIR)

## 입력 QC와 추론 규칙

- 24 kHz mono로 디코드하고, 길이가 30초 이상인 곡은 시작·중간·끝의 10초를 사용한다.
- `Logistic Regression`, `RBF-SVM`, `Log-Mel CNN`, `Frozen MERT + LR` 네 모델을 비교한다.
- Validation에서 고정한 track threshold를 적용한다. SVM은 확률이 아닌 decision margin이므로 숫자를 확률(%)로 해석하지 않는다.

In [ ]:
display(results["input_metadata"][["file_name", "duration_sec", "source_sample_rate_hz", "channels", "bytes", "average_bitrate_bps", "sha256"]])
display(results["track_predictions"][["file_name", "model", "n_segments", "ai_score", "track_threshold", "prediction"]].round(6))
display(results["consensus"])

In [ ]:
segment_view = results["segment_predictions"][["file_name", "segment_role", "start_sec", "model", "score", "segment_threshold", "segment_prediction"]].copy()
display(segment_view.round(6))

## 판정 결과

두 입력 모두 네 모델이 3개 구간을 일관되게 `AI 생성`으로 판정했다. 따라서 이번 두 파일에 대한 모델 합의 결과는 **4/4 모델 AI 생성**이다.

이는 두 곡에 대한 단일 입력 시연 결과이지, 새로운 데이터셋에서의 성능평가나 100% 확정 판정은 아니다. 특히 입력 파일은 16 kHz mono, 약 39–40 kbps MP3라 학습 분포와 codec 조건이 다를 수 있다. 자세한 수치와 재현 방법은 `docs/23_post_presentation_feedback.md`에 기록했다.